### Load and Inspect GRID3 dataset

In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv(r'C:\Users\USER\Desktop\Refpath\data\raw\grid3_nga_-_health_facilities_-1.csv')
df.head()

,X,Y,FID,globalid,uniq_id,timestamp,editor,latitude,longitude,wardname,...,statecode,updated_on,accessblty,func_stats,category,ownership,type,source,alt_name,prmry_name
0,13.148320,11.822320,1,af719462-abfd-4f47-9dc3-0987164e75ac,1,2020/07/04 00:00:00+00,tosin.williams,11.822320,13.148320,Maisandari,...,BR,2019/03/01 00:00:00+00,,Unknown,Primary Health Center,Private,Primary,eHA Polio,Nursing Home,G R A Nursing Home
1,13.147176,11.821615,2,a29b0328-d844-4358-b0ab-2e120b8fb30f,2,2020/07/04 00:00:00+00,mokobia.chidinma,11.821615,13.147176,Maisandari,...,BR,2019/03/01 00:00:00+00,Unknown,Functional,Primary Health Center,National Primary Healthcare Development Agency,Primary,Measles Campaign,Nursing Home,Gishili Health Center
2,12.816284,10.707092,3,b685b769-5c83-4f83-a182-00e7e1b777d8,3,2020/07/04 00:00:00+00,mokobia.chidinma,10.707092,12.816284,Ngohi,...,BR,2019/03/01 00:00:00+00,,Partially Functional,Primary Health Center,State Primary Healthcare Development Agency,Primary,eHA Polio,,Lehobi Primary Health Care
3,12.181628,10.615220,4,78e64f7a-cbb8-4357-9e64-a7e502534527,4,2020/07/04 00:00:00+00,tosin.williams,10.615220,12.181628,Galdimari,...,BR,2019/03/01 00:00:00+00,Unknown,Not Functional,Primary Health Center,Others,Primary,eHA Polio,Mandara Girau Dispensary,Dugja Idp Camp
4,13.313358,10.759527,5,409c97ce-7490-4dc2-a8f8-2b8d53ad2b12,5,2020/07/04 00:00:00+00,tosin.williams,10.759527,13.313358,Ngurthavu Kopa,...,BR,2019/03/01 00:00:00+00,Unknown,Partially Functional,Dispensary,Others,Primary,eHA Polio,,Kopa Maikudiri Dispensary


In [3]:
print(df.shape)

(46146, 24)


In [4]:
print(df.dtypes)

X             float64
Y             float64
FID             int64
globalid       object
uniq_id         int64
timestamp      object
editor         object
latitude      float64
longitude     float64
wardname       object
wardcode       object
lganame        object
lgacode         int64
statename      object
statecode      object
updated_on     object
accessblty     object
func_stats     object
category       object
ownership      object
type           object
source         object
alt_name       object
prmry_name     object
dtype: object


In [5]:
print(df.isnull().sum())

X                0
Y                0
FID              0
globalid         0
uniq_id          0
timestamp        0
editor           0
latitude         0
longitude        0
wardname         0
wardcode         0
lganame          0
lgacode          0
statename        0
statecode     1126
updated_on     223
accessblty       0
func_stats       0
category         0
ownership        0
type             0
source           0
alt_name         0
prmry_name       0
dtype: int64


### Drop irrelevant columns

In [6]:
cols_to_keep = [
    'prmry_name', 'latitude', 'longitude',
    'statename', 'lganame', 'wardname',
    'func_stats', 'category', 'type', 'ownership'
]

df = df[cols_to_keep]

# Rename for clarity
df.rename(columns={
    'prmry_name': 'name',
    'func_stats': 'functional_status',
    'type': 'care_level',
    'lganame': 'lga',
    'statename': 'state',
    'wardname': 'ward'
}, inplace=True)

#### Filter out non-functional facilities

In [9]:
df = df[df['functional_status'] == 'Functional']
print(f"Functional facilities: {len(df)}")

Functional facilities: 34282


#### Clean the care level column

In [11]:
def clean_care_level(val):
    val = str(val).strip().lower()
    if 'tertiary' in val:
        return 'tertiary'
    elif 'secondary' in val:
        return 'secondary'
    elif 'primary' in val or 'health post' in val:
        return 'primary'
    else:
        return 'primary'

df['care_level'] = df['care_level'].apply(clean_care_level)
print(df['care_level'].value_counts())

care_level
primary      32258
secondary     1229
tertiary       795
Name: count, dtype: int64


#### Cross-validate category against care level

In [ ]:
def override_care_level(row):
    cat = str(row['category']).lower()
    if any(x in cat for x in ['teaching hospital', 'federal medical center', 'specialist hospital']):
        return 'tertiary'
    elif any(x in cat for x in ['general hospital', 'cottage hospital', 'district hospital', 'comprehensive health center']):
        return 'secondary'
    elif any(x in cat for x in ['primary health center', 'dispensary', 'maternity home', 'community health']):
        return 'primary'
    else:
        return row['care_level']  # keep existing if no match

df['care_level'] = df.apply(override_care_level, axis=1)

#### Drop duplicates

In [14]:
before = len(df)
df.drop_duplicates(subset=['name', 'latitude', 'longitude'], inplace=True)
after = len(df)
print(f"Removed {before - after} duplicates")

Removed 0 duplicates


#### Remove irrelevant facility types e.g Pharmacies, veterinary clinics, laboratories

In [15]:
exclude = ['pharmacy', 'veterinary clinic', 'laboratory', 'research hospital']
df = df[~df['category'].str.lower().isin(exclude)]

#### Standardize state names

In [17]:
df['state'] = df['state'].str.strip().str.title()
df['lga'] = df['lga'].str.strip().str.title()

In [18]:
# Verify all 36 states + FCT are present
print(df['state'].unique())

['Borno' 'Bauchi' 'Ogun' 'Zamfara' 'Kogi' 'Yobe' 'Abia' 'Ebonyi' 'Edo'
 'Imo' 'Ekiti' 'Kebbi' 'Anambra' 'Osun' 'Rivers' 'Sokoto' 'Benue'
 'Adamawa' 'Gombe' 'Bayelsa' 'Ondo' 'Lagos' 'Oyo' 'Enugu' 'Kaduna'
 'Nasarawa' 'Delta' 'Cross River' 'Fct' 'Katsina' 'Kwara' 'Plateau'
 'Jigawa' 'Taraba' 'Kano' 'Niger' 'Akwa Ibom']


#### Add GeoJSON facilities as supplement

In [23]:
import json

with open(r"C:\Users\USER\Desktop\Refpath\data\raw\health_facilities.geojson") as f:
    geo = json.load(f)

In [27]:
geo_records = []
for feature in geo['features']:
    props = feature['properties']
    geometry = feature.get('geometry')
    
    # Skip features with missing or malformed geometry
    if not geometry:
        continue
    coords = geometry.get('coordinates')
    if not coords or len(coords) < 2:
        continue
    
    geo_records.append({
        'name': props.get('name'),
        'latitude': coords[1],
        'longitude': coords[0],
        'state': props.get('adm1_name'),
        'lga': props.get('adm2_name'),
        'ward': props.get('adm3_name'),
        'functional_status': 'Functional',
        'category': props.get('healthcare') or props.get('amenity'),
        'care_level': None,
        'ownership': props.get('operator_type')
    })

geo_df = pd.DataFrame(geo_records)

In [28]:
geo_df.dropna(subset=['name', 'latitude', 'longitude'], inplace=True)
geo_df.head()

,name,latitude,longitude,state,lga,ward,functional_status,category,care_level,ownership
0,Alaso Amikanle Primary Health Care Center,6.650362,3.244609,Lagos,Alimosho,None,Functional,hospital,None,public
1,Sholay Beauty World,6.536059,3.220142,Lagos,Alimosho,None,Functional,hospital,None,None
2,Marie Stopes Clinic General Medical Care And L...,6.5405,3.223657,Lagos,Alimosho,None,Functional,hospital,None,None
3,Jubilee Health Center And Laboratory,6.535851,3.22721,Lagos,Alimosho,None,Functional,hospital,None,None
4,Adebek Maternity Home,6.525849,3.216382,Lagos,Alimosho,None,Functional,hospital,None,None


#### Infer care level from name since category is sparse in OSM

In [41]:
def infer_from_name(name):
    name = str(name).lower()
    if any(x in name for x in ['teaching', 'specialist', 'federal medical']):
        return 'tertiary'
    elif any(x in name for x in ['general hospital', 'cottage', 'comprehensive']):
        return 'secondary'
    else:
        return 'primary'

geo_df['care_level'] = geo_df['name'].apply(infer_from_name)

#### Clean coordinates in geo_df before merging

In [31]:
# Clean coordinates in geo_df before merging
geo_df['latitude'] = pd.to_numeric(geo_df['latitude'], errors='coerce')
geo_df['longitude'] = pd.to_numeric(geo_df['longitude'], errors='coerce')
geo_df.dropna(subset=['latitude', 'longitude'], inplace=True)

In [32]:
df['latitude'] = pd.to_numeric(df['latitude'], errors='coerce')
df['longitude'] = pd.to_numeric(df['longitude'], errors='coerce')
df.dropna(subset=['latitude', 'longitude'], inplace=True)

#### Merge teh datasets

In [33]:
#merge the datasets
combined = pd.concat([df, geo_df], ignore_index=True)
combined.drop_duplicates(subset=['name', 'latitude', 'longitude'], inplace=True)
print(f"Combined dataset: {len(combined)} facilities")

Combined dataset: 37261 facilities


#### Check for coverage and distribution

In [34]:
# Check coverage and distribution
print("=== Care Level Distribution ===")
print(combined['care_level'].value_counts())

print("\n=== States Coverage ===")
print(f"States covered: {combined['state'].nunique()}")
print(combined['state'].value_counts())

print("\n=== Null Check ===")
print(combined.isnull().sum())

print("\n=== Sample Rows ===")
print(combined[['name', 'state', 'lga', 'care_level', 'latitude', 'longitude']].head(10).to_string())

=== Care Level Distribution ===
care_level
primary      35070
secondary     1311
tertiary       880
Name: count, dtype: int64

=== States Coverage ===
States covered: 38
state
Lagos                        2623
Benue                        1902
Niger                        1815
Kano                         1723
Katsina                      1482
Osun                         1446
Oyo                          1376
Cross River                  1313
Ogun                         1293
Anambra                      1280
Borno                        1230
Imo                          1215
Kaduna                       1199
Abia                         1162
Kogi                         1123
Enugu                        1059
Bauchi                       1002
Nasarawa                      938
Taraba                        908
Edo                           889
Ebonyi                        859
Adamawa                       843
Ondo                          820
Akwa Ibom                     792
Sokoto  

#### Merge FCT duplicates

In [38]:
combined['state'] = combined['state'].replace('Fct', 'Federal Capital Territory')

In [39]:
# Verify
print(combined['state'].nunique())
print(combined[combined['state'] == 'Federal Capital Territory'].shape)

37
(518, 10)


In [ ]:
final_cols = ['name', 'latitude', 'longitude', 'state', 'lga', 'care_level', 'category', 'functional_status']
combined = combined[final_cols]
combined.to_csv("facilities_clean.csv", index=False)
print(f"Saved {len(combined)} facilities")

#### Save the cleaned dataset

In [40]:
final_cols = ['name', 'latitude', 'longitude', 'state', 'lga', 'care_level', 'category', 'functional_status']
combined = combined[final_cols]
combined.to_csv("facilities_clean.csv", index=False)
print(f"Saved {len(combined)} facilities")

Saved 37261 facilities
